In [3]:
"""
conda activate your_env_name

# Remove problematic installations
pip uninstall numpy opencv-python -y
conda uninstall numpy opencv -y

# Install stable versions from conda-forge
conda install -c conda-forge numpy opencv

pip uninstall face_recognition -y
pip uninstall dlib -y
conda install -c conda-forge dlib
pip install face_recognition
"""

'\nconda activate your_env_name\n\n# Remove problematic installations\npip uninstall numpy opencv-python -y\nconda uninstall numpy opencv -y\n\n# Install stable versions from conda-forge\nconda install -c conda-forge numpy opencv\n\npip uninstall face_recognition -y\npip uninstall dlib -y\nconda install -c conda-forge dlib\npip install face_recognition\n'

In [1]:
import cv2
import face_recognition
import dlib

print(face_recognition.__version__)
print(cv2.__version__)
print(dlib.__version__)


1.2.3
4.11.0
19.24.6


In [16]:
sample_interval = 600

In [ ]:
import os
import cv2

def extract_frames(video_path, interval_sec=sample_interval):
    vidcap = cv2.VideoCapture(video_path)
    frames = []
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps * interval_sec)

    success, image = vidcap.read()
    count = 0
    while success:
        if count % frame_interval == 0:
            frames.append(image)
        success, image = vidcap.read()
        count += 1
    vidcap.release()
    return frames

import face_recognition

def recognize_faces(image, known_encodings, known_names):
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    encodings = face_recognition.face_encodings(rgb_image)

    names = []
    for encoding in encodings:
        matches = face_recognition.compare_faces(known_encodings, encoding)
        if True in matches:
            first_match_index = matches.index(True)
            names.append(known_names[first_match_index])
        else:
            names.append("Unknown")
    return names

def create_face_encoding(image_path):
    image = face_recognition.load_image_file(image_path)
    encoding = face_recognition.face_encodings(image)[0]
    return encoding


def extract_encodings_from_video(video_path, frame_interval=sample_interval):
    """Extract face encodings from a video file at regular intervals (seconds)."""
    video_capture = cv2.VideoCapture(video_path)
    encodings = []
    fps = video_capture.get(cv2.CAP_PROP_FPS)
    frame_gap = int(fps * frame_interval)

    frame_count = 0
    success, frame = video_capture.read()
    while success:
        if frame_count % frame_gap == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            face_encs = face_recognition.face_encodings(rgb_frame)
            encodings.extend(face_encs)
        success, frame = video_capture.read()
        frame_count += 1

    video_capture.release()
    return encodings

def build_known_faces_encodings(root_dir, max_folders=0, max_videos_per_folder=5):
    """Build known face encodings from sub-folders containing MP4 videos, sampling at most 10 videos per first-layer sub-folder."""
    known_encodings = []
    known_names = []

    for name in os.listdir(root_dir)[:max_folders] if max_folders else os.listdir(root_dir):
        person_folder = os.path.join(root_dir, name)
        if not os.path.isdir(person_folder):
            continue
        videos_processed = 0
        for subdir, _, files in os.walk(person_folder):
            for video_file in files:
                if video_file.lower().endswith(".mp4"):
                    if videos_processed >= max_videos_per_folder:
                        break
                    video_path = os.path.join(subdir, video_file)
                    encodings = extract_encodings_from_video(video_path)
                    known_encodings.extend(encodings)
                    known_names.extend([name] * len(encodings))
                    videos_processed += 1
            if videos_processed >= max_videos_per_folder:
                break

    return known_encodings, known_names

In [ ]:
root_directory = "H:/recov/collections"
known_encodings, known_names = build_known_faces_encodings(root_directory)
print(f"Built encodings for {len(set(known_names))} known individuals.")

KeyboardInterrupt: 

In [ ]:

video_path = "your_video.mp4"
frames = extract_frames(video_path)

for frame in frames:
    names = recognize_faces(frame, known_encodings, known_names)
    print("Recognized:", names)


In [19]:
import os
import cv2
import face_recognition
import torch

def extract_encodings_from_video(video_path, frame_interval=sample_interval):
    """Extract face encodings from a video file at regular intervals (seconds) using GPU."""
    video_capture = cv2.VideoCapture(video_path)
    encodings = []
    fps = video_capture.get(cv2.CAP_PROP_FPS)
    frame_gap = int(fps * frame_interval)

    frame_count = 0
    success, frame = video_capture.read()
    while success:
        if frame_count % frame_gap == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            face_locations = face_recognition.face_locations(rgb_frame, model='cnn')
            face_encs = face_recognition.face_encodings(rgb_frame, face_locations)
            encodings.extend(face_encs)
        success, frame = video_capture.read()
        frame_count += 1

    video_capture.release()
    return encodings

def build_known_faces_encodings(root_dir, max_videos_per_folder=1):
    """Build known face encodings from sub-folders containing MP4 videos, sampling at most 10 videos per first-layer sub-folder using GPU."""
    known_encodings = []
    known_names = []

    for name in os.listdir(root_dir):
        person_folder = os.path.join(root_dir, name)
        if not os.path.isdir(person_folder):
            continue
        videos_processed = 0
        for subdir, _, files in os.walk(person_folder):
            for video_file in files:
                if video_file.lower().endswith(".mp4"):
                    if videos_processed >= max_videos_per_folder:
                        break
                    print(f"Processing {video_file}...")
                    video_path = os.path.join(subdir, video_file)
                    encodings = extract_encodings_from_video(video_path)
                    known_encodings.extend(encodings)
                    known_names.extend([name] * len(encodings))
                    videos_processed += 1
            if videos_processed >= max_videos_per_folder:
                break

    return known_encodings, known_names

# Example usage
if __name__ == "__main__":
    if not torch.cuda.is_available():
        print("Warning: GPU not available. The script will run on CPU.")
    else:
        print("GPU detected. Running on GPU.")

    root_directory = "D:/Downloads"
    known_encodings, known_names = build_known_faces_encodings(root_directory)

    print(f"Built encodings for {len(set(known_names))} known individuals.")


Processing SIVR-001.mp4...
Processing ABW-097-UC.mp4...


KeyboardInterrupt: 